In [2]:
# Etape 1 - du JSON au DataFrame
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

import pandas as pd

df = pd.json_normalize(data,
                                   record_path="restaurants",
                                   meta=["ville", "pays"])

df_restaurants = df.drop(columns="notes")

df_restaurants

,nom,adresse,type,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie


In [3]:
# Etape 2 - la note moyenne

df_restaurants["note_moyenne"] = df["notes"].apply(lambda x: round(sum(x) / len(x), 2) if isinstance(x, list) and len(x) > 0 else None)

df_restaurants

,nom,adresse,type,ville,pays,note_moyenne
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,4.70
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,4.45
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,4.15
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,4.00


In [4]:
# Etape 3 - géocoder
import time
import requests

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (bourreauclemence@gmail.com)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [5]:
df_restaurants[["lat", "lon"]] = df_restaurants["adresse"].apply(
    lambda x: pd.Series(geocoder_monde(x))
)

df_restaurants

,nom,adresse,type,ville,pays,note_moyenne,lat,lon
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,4.70,37.502093,15.093195
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,4.45,37.506171,15.094322
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,4.15,37.489569,15.077596
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,4.00,37.502933,15.093041


In [10]:
# Etape 4 - la carte
import folium

# Pour déterminer la zone à afficher
coordonnees = [df_restaurants["lat"].mean(), df_restaurants["lon"].mean()]

# Création initiale de la carte
carte = folium.Map(location=coordonnees, zoom_start=14)

# Pour ajouter les points sur la carte
for _, ligne in df_restaurants.iterrows():
    if pd.notna(ligne["lat"]):
        folium.Marker(
            location=[ligne["lat"], ligne["lon"]],
            popup=f"{ligne["nom"]} - note moyenne : {ligne["note_moyenne"]}",
        ).add_to(carte)

carte

In [6]:
# Etape 5 - garder le résultat
df_restaurants.to_csv("../data/restaurants_geocodes.csv", encoding="utf-8", index=False)